# Carbon and Consequence
## Explainer Notebook

**02806 Social Data Analysis and Visualization — DTU**  
Akash Neil Das (s252577) · Ketan Nandkumar Patil (s252607)

---

This notebook contains all the behind-the-scenes analysis supporting the final project website. It covers motivation, dataset description, data cleaning, exploratory analysis, machine learning, genre choices, and a critical discussion of what worked and what could be improved.


## 1. Motivation

### What is the dataset?

Our analysis is built upon the Our World in Data CO₂ and Greenhouse Gas Emissions dataset, a rigorous meta-collection that synthesizes data from the Global Carbon Project, NASA GISS, and the International Energy Agency. 

This repository is exceptional in its breadth, spanning 254 countries and territories with a temporal reach that begins in 1750 and extends to 2024. By integrating 79 distinct variables—ranging from fuel-specific emissions like coal and gas to macroeconomic indicators such as GDP and population—the dataset allows for a multi-dimensional view of how human activity translates into atmospheric change.

### Why did we choose this dataset?

We selected this specific dataset because climate change is the defining challenge of our generation, yet most public discourse lacks a concrete intuition for the underlying figures. While many datasets focus strictly on climate metrics, this collection is uniquely rich because it bridges the gap between environmental science and socio-economics. It enables us to move beyond simple totals and address complex questions regarding historical fairness, the link between national wealth and carbon intensity, and the long-term trajectory of global warming. 

Furthermore, the credibility of the sourcing—relying on peer-reviewed scientific institutions—provides a factual foundation that is essential when navigating a topic that is often politically charged.

### What was the goal for the end user's experience?

Our target audience consists of DTU students and peers who are aware of the climate crisis but may not have engaged deeply with the raw data. The goal for the end-user experience was to distill vast amounts of information into three core realizations. First, we wanted users to distinguish between total national responsibility and per-capita footprints. 

Second, we aimed to demonstrate that while wealth and emissions are historically correlated, policy choices can decouple economic growth from environmental harm. Finally, we wanted to provide a clear, model-driven sense of where the planet is headed if current trends persist.

To ensure the project remains accessible, we intentionally designed the final website to be non-technical and intuitive. We made the deliberate choice to omit model equations, p-values, and domain-specific jargon that might create a barrier to entry for a general audience. 

By prioritizing clean visualizations and clear narratives on the front end, we allow the data to speak for itself. The technical rigor, complex data cleaning, and statistical validation are purposely contained within this explainer notebook, serving as the "engine room" for the streamlined user experience.


## 2. Basic Stats


In [17]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('owid-co2-data.csv')
print('Full dataset shape:', df.shape)
print('Year range:', df['year'].min(), 'to', df['year'].max())
print('Countries:', df['country'].nunique())


Full dataset shape: (50411, 79)
Year range: 1750 to 2024
Countries: 254


In [18]:
import pandas as pd

df = pd.read_csv('owid-co2-data.csv')

total_rows = len(df)
total_cols = len(df.columns)
year_range = f"{df['year'].min()} - {df['year'].max()}"
unique_entities = df['country'].nunique()

completion_df = pd.DataFrame({
    'Variable': df.columns,
    'Observations': df.notnull().sum(),
    'Coverage (%)': ((df.notnull().sum() / total_rows) * 100).round(2)
}).sort_values(by='Coverage (%)', ascending=False)

print(f"Dataset Dimensions: {total_rows} rows x {total_cols} columns")
print(f"Time Period: {year_range}")
print(f"Total Unique Regions/Countries: {unique_entities}")
print("\n--- Top 10 Most Complete Variables ---")
print(completion_df.head(10).to_string(index=False))
print("\n--- Bottom 10 Most Complete Variables ---")
print(completion_df.tail(10).to_string(index=False))

Dataset Dimensions: 50411 rows x 79 columns
Time Period: 1750 - 2024
Total Unique Regions/Countries: 254

--- Top 10 Most Complete Variables ---
                            Variable  Observations  Coverage (%)
                             country         50411        100.00
                                year         50411        100.00
                            iso_code         42480         84.27
         temperature_change_from_ghg         41238         81.80
         temperature_change_from_co2         41238         81.80
share_of_temperature_change_from_ghg         41238         81.80
                          population         41167         81.66
                       nitrous_oxide         38500         76.37
         temperature_change_from_n2o         38280         75.94
         temperature_change_from_ch4         38280         75.94

--- Bottom 10 Most Complete Variables ---
                         Variable  Observations  Coverage (%)
                  consumption_co2  

### Data Cleaning and Preprocessing

Our cleaning choices were as follows:

1. **Filter to 1990 onwards.** Data before 1990 has significant gaps, especially for economic variables like GDP. Filtering gives us 8,883 rows with much higher completeness on the key columns we care about.
2. **Remove aggregated regions.** The dataset includes rows for World, Asia, Europe, etc. These use an `iso_code` starting with `OWID_` or have no ISO code. We keep them for global trend analysis but exclude them from country-level charts.
3. **No imputation.** We dropped rows with missing values on a per-chart basis rather than imputing. Given that missingness is mostly structural (some countries did not report certain data in certain years), imputation would introduce more noise than it removes.
4. **GDP per capita.** This variable is not directly in the dataset but can be computed as `gdp / population`. We create it on the fly for the scatter plot.


In [19]:

df = df[df['year'] >= 1990].copy()
countries = df[~df['iso_code'].isna() & ~df['iso_code'].str.startswith('OWID', na=False)].copy()
world = df[df['country'] == 'World'].copy()
print('After 1990 filter:', df.shape)
print('Country rows:', countries.shape)
key_cols = ['co2', 'co2_per_capita', 'gdp', 'population', 
            'coal_co2', 'oil_co2', 'gas_co2', 'temperature_change_from_co2']
completeness = (countries[key_cols].notna().mean() * 100).round(1)
print('\nColumn completeness (% non-null):')
print(completeness.to_string())

After 1990 filter: (8883, 79)
Country rows: (7630, 79)

Column completeness (% non-null):
co2                            98.4
co2_per_capita                 97.5
gdp                            70.9
population                     99.1
coal_co2                       62.3
oil_co2                        97.7
gas_co2                        55.8
temperature_change_from_co2    98.6


### Exploratory Data Analysis


In [20]:
# Global CO2 trend
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#F8FAFC')

# Left: world CO2 over time
ax = axes[0]
ax.set_facecolor('white')
ax.plot(world['year'], world['co2'], color='#1E3A5F', lw=2.5)
ax.fill_between(world['year'], world['co2'], alpha=0.08, color='#1E3A5F')
ax.set_title('Global CO2 Emissions, 1990 to 2022', fontsize=12, pad=10)
ax.set_xlabel('Year')
ax.set_ylabel('Million tonnes CO2')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for spine in ax.spines.values(): spine.set_visible(False)
ax.grid(axis='y', color='#E2E8F0', lw=0.8)

# Right: top 10 emitters 2022
ax2 = axes[1]
ax2.set_facecolor('white')
top10 = countries[countries['year']==2022].nlargest(10,'co2')[['country','co2']].dropna()
colors = ['#C0392B' if i==0 else '#1E3A5F' for i in range(len(top10))]
ax2.barh(top10['country'][::-1], top10['co2'][::-1], color=colors[::-1])
ax2.set_title('Top 10 Emitters, 2022', fontsize=12, pad=10)
ax2.set_xlabel('Million tonnes CO2')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for spine in ax2.spines.values(): spine.set_visible(False)
ax2.grid(axis='x', color='#E2E8F0', lw=0.8)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=130, bbox_inches='tight', facecolor='#F8FAFC')
plt.show()
print(f"CO2 1990: {world[world.year==1990].co2.values[0]:,.0f} Mt")
print(f"CO2 2022: {world[world.year==2022].co2.values[0]:,.0f} Mt")


CO2 1990: 22,732 Mt
CO2 2022: 37,528 Mt


In [21]:
# Distribution of CO2 per capita across countries
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#F8FAFC')
ax.set_facecolor('white')

data_2022 = countries[countries['year']==2022]['co2_per_capita'].dropna()
ax.hist(data_2022, bins=40, color='#1E3A5F', edgecolor='white', linewidth=0.5)
ax.axvline(data_2022.median(), color='#EF4444', lw=2, ls='--',
           label=f'Median: {data_2022.median():.1f} t')
ax.axvline(data_2022.mean(), color='#F59E0B', lw=2, ls='--',
           label=f'Mean: {data_2022.mean():.1f} t')
ax.set_title('Distribution of CO2 per Capita Across Countries, 2022', fontsize=12)
ax.set_xlabel('CO2 per capita (tonnes)')
ax.set_ylabel('Number of countries')
ax.legend()
for spine in ax.spines.values(): spine.set_visible(False)
ax.grid(axis='y', color='#E2E8F0', lw=0.8)
plt.tight_layout()
plt.savefig('eda_distribution.png', dpi=130, bbox_inches='tight', facecolor='#F8FAFC')
plt.show()
print(f'Median CO2 per capita: {data_2022.median():.2f} tonnes')
print(f'Mean CO2 per capita: {data_2022.mean():.2f} tonnes')
print(f'Max (Qatar): {data_2022.max():.2f} tonnes')


Median CO2 per capita: 3.16 tonnes
Mean CO2 per capita: 4.69 tonnes
Max (Qatar): 37.89 tonnes


## 3. Data Analysis

### Key findings from the exploratory analysis

1. **Global CO₂ grew 65% from 1990 to 2022**, from 22,732 to 37,528 million tonnes. The trend is broadly linear with a dip during the 2008 financial crisis and a sharp drop in 2020 due to COVID-19 lockdowns, followed by a rapid rebound.

2. **China's growth dominates the story**. It went from around 2,500 Mt in 1990 to nearly 12,000 Mt in 2022, now accounting for roughly 31% of global emissions. No other country comes close to this trajectory.

3. **Rich countries have largely plateaued or declined**. The US peaked around 2005 and has declined since, largely through the shale gas revolution displacing coal and growing renewable penetration. EU countries show a consistent downward trend.

4. **Per capita emissions are highly unequal**. The distribution across countries is heavily right-skewed. The median country emits around 3 to 4 tonnes per person, but Gulf states and some industrialised nations emit 15 to 30 tonnes. This inequality is important for any discussion of climate justice.

5. **GDP and emissions are correlated but not deterministic**. The scatter plot shows a clear positive relationship on a log scale, but the variance is wide. France emits roughly half as much per capita as Germany despite similar GDP levels, due to nuclear energy. Brazil and Indonesia are large economies with relatively low per capita emissions.


## Machine Learning: Linear Regression for 2030 Prediction


In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# Prepare data
world_ml = df[df['country']=='World'][['year','co2']].dropna().sort_values('year')
X = world_ml['year'].values.reshape(-1, 1)
y = world_ml['co2'].values

# Fit model
reg = LinearRegression()
reg.fit(X, y)

# Evaluate
y_pred = reg.predict(X)
r2 = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)

print(f'R² score: {r2:.4f}')
print(f'MAE: {mae:,.0f} million tonnes')
print(f'Slope: {reg.coef_[0]:,.0f} Mt per year')
print(f'Intercept: {reg.intercept_:,.0f}')

# Predict 2023 to 2030
future = np.arange(2023, 2031).reshape(-1, 1)
preds = reg.predict(future)
for yr, val in zip(range(2023, 2031), preds):
    print(f'  {yr}: {val:,.0f} Mt')


R² score: 0.9622
MAE: 902 million tonnes
Slope: 534 Mt per year
Intercept: -1,041,725
  2023: 38,969 Mt
  2024: 39,503 Mt
  2025: 40,038 Mt
  2026: 40,572 Mt
  2027: 41,106 Mt
  2028: 41,640 Mt
  2029: 42,174 Mt
  2030: 42,709 Mt


### Model discussion

We chose **ordinary least squares linear regression** for several reasons:

1. **The historical data is nearly linear**. With an R² of 0.962, a linear model fits the 1990 to 2022 trend extremely well. More complex models would overfit without adding meaningful predictive power for a short 8-year forecast horizon.
2. **Interpretability**. The slope coefficient (534 Mt per year) is directly interpretable: global emissions have grown by about 534 million tonnes every year on average since 1990. This is a useful number to communicate to a non-technical audience.
3. **Honest baseline**. Linear regression gives a business-as-usual scenario. It does not assume any acceleration or deceleration. This makes it a fair baseline against which policy targets can be compared.

**Limitations**:
- Our model operates on a "momentum" principle—the idea that the future is simply an extension of the past. However, history is rarely a straight line; it is a series of structural breaks. A statistical model cannot "see" a sudden carbon tax passed by a major economy, a breakthrough in fusion energy, or a global pandemic that permanently shifts commuting habits. By treating the next eight years as a continuation of the last thirty, we essentially assume that no revolutionary policy or technology will intervene to bend the curve.

- By using "Year" as our sole predictor, we have built a model that is remarkably clear but intentionally narrow. In the real world, emissions are a complex byproduct of population growth, GDP fluctuations, and the specific "carbon intensity" of a nation's energy mix. A more sophisticated model would treat these as distinct gears in a machine. Our choice to stick to a single feature reflects our design philosophy: we wanted to show the current heading of the ship, even if we aren't accounting for every current and wind speed that might push it off course.

- The confidence intervals displayed on our website provide a sense of security that is, in truth, somewhat fragile. These margins are based on "residual standard deviation"—essentially, how much the historical data points wiggled around the line in the past. While this gives us a measure of historical fit, it drastically underestimates "forecast uncertainty." In truth, as we look further into the future (an 8-year horizon), the cone of possibility widens significantly. The model tells us where we might go if everything remains equal, but the true uncertainty of the 2030s is much broader than any shaded region on a graph can capture. 

## 4. Genre

### Which genre of data story did we use?

While our project shares the linear flow of a long-form article, it is technically categorized as a Partitioned Poster (Segel and Heer, 2010). Unlike a traditional "Magazine Style" layout where images are secondary to a body of text, our data story is structured into discrete, high-contrast visual modules

This modular approach allows for a "multi-view" experience where the user can process complex scientific information in manageable segments before moving to the next act of the narrative.

### Visual Narrative tools

**Visual Structuring:**
- *Consistent visual platform*: all four charts share the same colour palette, font family, and background colour, creating a coherent visual identity across the page
- *Establishing shot*: the animated world map at the top functions as an establishing shot, giving the reader an immediate global overview before we zoom into specific countries and questions

**Highlighting:**
- *Feature distinction*: China is highlighted in red in the line chart to draw attention to the most significant trend
- *Annotations*: the 2030 prediction chart uses an annotation to call out the specific predicted value

**Transition Guidance:**
- *Object continuity*: the same countries appear across multiple charts, letting readers carry their understanding from one visualisation to the next
- *Animated transitions*: the choropleth map uses animation across years as a transition device

### Narrative Structure tools

**Ordering:**
- *Our project follows the Martini Glass narrative structure* : the most common hybrid model in data storytelling. The "stem" of the glass is represented by our strictly linear ordering , where the author guides the reader through a sequence of pre-determined visualizations. This ensures that the foundational context—the scale of emissions—is understood before the user is invited to explore more complex economic relationships

**Interactivity:**
- *Hover highlighting and details on demand*: all four charts support hover for exact values
- *Filtering and selection*: the line chart legend allows toggling individual countries; the choropleth has a year slider
- *Very limited interactivity*: we kept interactivity constrained so it enriches rather than distracts from the narrative

**Messaging:**
- *Captions*: every visualisation has a detailed caption explaining what to look at and why it matters
- *Introductory text*: each section opens with text that frames the question before presenting the chart
- *Accompanying article*: the full website functions as an article that could stand alone without the charts


## 5. Visualizations

### Figure 1: Animated choropleth map (CO₂ per capita)
**Why this chart:** A choropleth map is the natural choice for country-level data with a geographic dimension. Animation across years lets the reader see change without requiring them to compare multiple static maps. Per capita (rather than total) normalises for country size, making the comparison fair across countries as different as China and Qatar.

### Figure 2: Multi-line chart (top 10 emitters)
**Why this chart:** Total emissions by country is the most politically relevant metric for climate negotiations. A line chart over time shows trajectory, not just snapshot. The interactive legend lets readers isolate specific countries they are interested in. China's trajectory is so dominant that it needed to be shown against others to communicate its scale.

### Figure 3: Bubble scatter plot (GDP vs CO₂ per capita)
**Why this chart:** The GDP-emissions relationship is one of the most debated questions in climate economics. A scatter plot directly shows the correlation and the variance around it. Bubble size encodes population (a third relevant variable) without adding a separate chart. Colour by dominant fuel type adds a fourth dimension that helps explain outliers.

### Figure 4: Linear regression prediction
**Why this chart:** A prediction chart directly answers the most forward-looking question in the story: where are we headed? 
Showing the historical line alongside the projection gives the model credibility. The confidence band communicates uncertainty honestly. The contrast with the Paris Agreement target (mentioned in the text) gives the reader a reference point.


## 6. Discussion

### What went well
- The dataset was exceptionally clean and well-documented, which meant most of our time went into analysis and storytelling rather than data wrangling
- The four-chart structure worked well: each chart answers a distinct question and the sequence feels natural
- The linear regression model fit the historical data remarkably well (R² = 0.962), giving us a credible and clearly interpretable prediction
- The website design is clean and consistent, with a visual identity that feels appropriate for the topic

### What is still missing and could be improved
- **More sophisticated ML**: a multivariate model using GDP growth, energy mix, and population as features would produce more nuanced predictions. We could also model individual countries rather than global totals
- **Deforestation and land use**: the dataset includes land-use CO₂ figures but we did not incorporate them into the main charts. For countries like Brazil and Indonesia this is a significant omission
- **Scenario analysis**: the prediction currently shows only a business-as-usual scenario. Showing optimistic and pessimistic scenarios alongside would be more informative
- **Scrollytelling**: the page currently uses a standard scroll layout. A scrollytelling approach where charts animate as the user scrolls would create a more immersive experience
- **Mobile responsiveness**: the interactive Plotly charts do not resize perfectly on mobile screens

## 7. Contributions

- **Ketan Nandkumar Patil (s252607)**: led the data analysis,led interactive visualization construction,machine learning model and narrative writing. Wrote sections 5, 6, and 7 of this notebook.
- **Akash Neil Das (s252577)**: led the website design,led exploratory visualizations,led machine learning model. Wrote sections 2, 3, and 4 of this notebook.

